In [0]:
%sql
USE CATALOG db_healthcare_kl;
USE SCHEMA omop;

In [0]:
condition_occurrence_df = spark.table("condition_occurrence")

condition_occurrence_df.createOrReplaceTempView("condition_occurrence_view")

In [0]:
person_df = spark.table("person_data")

person_df.createOrReplaceTempView("person_data_view")


In [0]:
display(person_df)

In [0]:
%sql
CREATE OR REPLACE TABLE db_healthcare_kl.gold.patient_cohorts AS

select person_id, 'Diabetes' AS Cohort_name 
from condition_occurrence_view
where condition_name = 'Diabetes'

union all

SELECT
    person_id,
    'HyperTension' AS cohort_name
FROM condition_occurrence
WHERE condition_name = 'HyperTension'

union all
SELECT
    person_id,
    'Age_Above_60' AS cohort_name
FROM person_data_view
WHERE floor(months_between(current_date(), Date_of_birth)/12) >= 60



In [0]:
%sql
select * from db_healthcare_kl.gold.patient_cohorts

In [0]:
encounter_df=spark.table("visit_occurrence")

encounter_df.createOrReplaceTempView("encounter_occurrence_view")


In [0]:
display(encounter_df)

In [0]:
%sql
create or replace table db_healthcare_kl.gold.Encounter_summaries

SELECT
    person_id,
    COUNT(visit_occurrence_id) AS total_visits,
    MAX(datediff(visit_end_date, visit_start_date)) AS max_duration_days,
    AVG(datediff(visit_end_date, visit_start_date)) AS avg_duration_days
FROM encounter_occurrence_view
GROUP BY person_id

In [0]:
%sql
select * from db_healthcare_kl.gold.Encounter_summaries

In [0]:
%sql

CREATE OR REPLACE TABLE db_healthcare_kl.gold.readmission_within_30days AS

WITH visits AS (
    SELECT
        person_id,
        visit_occurrence_id,
        visit_start_date,
        visit_end_date,

        LEAD(visit_start_date) OVER (
            PARTITION BY person_id
            ORDER BY visit_start_date
        ) AS next_visit_start_date

    FROM visit_occurrence
)

SELECT
    person_id,
    visit_occurrence_id,
    visit_end_date,
    next_visit_start_date,

    CASE
        WHEN datediff(next_visit_start_date, visit_end_date) <= 30
             AND next_visit_start_date IS NOT NULL
        THEN 1
        ELSE 0
    END AS readmission_30day_flag

FROM visits;


In [0]:
%sql
SELECT * FROM db_healthcare_kl.gold.readmission_within_30days;

KPI 1 ADT Volume


In [0]:
%sql
CREATE OR REPLACE TABLE db_healthcare_kl.gold.kpi_adt_volume AS

SELECT
    date_format(visit_start_date, 'yyyy-MM') AS month,
    COUNT(*) AS total_visits

FROM db_healthcare_kl.omop.visit_occurrence

GROUP BY date_format(visit_start_date, 'yyyy-MM')

ORDER BY month;

In [0]:
%sql
select * from db_healthcare_kl.gold.kpi_adt_volume

KPI 2 Readmission Rate

In [0]:
%sql

CREATE OR REPLACE TABLE db_healthcare_kl.gold.kpi_readmission_rate AS

SELECT
    ROUND(
        (SUM(readmission_30day_flag) * 100.0) / COUNT(*),
        2
    ) AS readmission_rate
FROM db_healthcare_kl.gold.readmission_within_30days;

In [0]:
%sql

SELECT *
FROM db_healthcare_kl.gold.kpi_readmission_rate;

KPI 3 Top Diagnoses

In [0]:
%sql

CREATE OR REPLACE TABLE db_healthcare_kl.gold.kpi_top_diagnoses AS

SELECT
    condition_name AS diagnosis,
    COUNT(*) AS total_cases

FROM db_healthcare_kl.omop.condition_occurrence
WHERE condition_source_value IS NOT NULL
GROUP BY condition_name
ORDER BY total_cases DESC;

In [0]:
%sql

SELECT *
FROM db_healthcare_kl.gold.kpi_top_diagnoses
LIMIT 20;

KPI 4 Medication Usage

In [0]:
%sql
CREATE OR REPLACE TABLE db_healthcare_kl.gold.kpi_medication_usage AS

SELECT
    drug_name AS medication_name,
    COUNT(*) AS total_usage

FROM db_healthcare_kl.omop.drug_exposure

WHERE drug_name IS NOT NULL

GROUP BY drug_name

ORDER BY total_usage DESC

LIMIT 10;

In [0]:
%sql
select * from db_healthcare_kl.gold.kpi_medication_usage;

KPI 5 Length of stay

In [0]:
%sql
CREATE OR REPLACE TABLE db_healthcare_kl.gold.kpi_length_of_stay AS

SELECT
    CASE
        WHEN visit_type = 'AMB' THEN 'Ambulatory'
        WHEN visit_type = 'EMER' THEN 'Emergency'
        WHEN visit_type = 'IMP' THEN 'Inpatient'
        ELSE visit_type
    END AS visit_type,

    ROUND(AVG(DATEDIFF(visit_end_date, visit_start_date)), 2) AS avg_stay_days

FROM db_healthcare_kl.omop.visit_occurrence

WHERE visit_end_date IS NOT NULL

GROUP BY visit_type

ORDER BY avg_stay_days DESC;

In [0]:
%sql
select * from db_healthcare_kl.gold.kpi_length_of_stay;

In [0]:
%sql
SELECT
    date_format(visit_start_date,'yyyy-MM') AS month,
    COUNT(*) AS visits
FROM db_healthcare_kl.omop.visit_occurrence
WHERE date_format(visit_start_date,'yyyy-MM')='2014-03'
GROUP BY month;

In [0]:
%sql
SELECT *
FROM db_healthcare_kl.gold.kpi_adt_volume
WHERE month='2014-03';

In [0]:
%sql
SELECT COUNT(*) FROM db_healthcare_kl.silver.encounter_silver_dlt;

In [0]:
%sql
SELECT COUNT(*) FROM db_healthcare_kl.omop.visit_occurrence;

In [0]:
%sql
SELECT
date_format(start_date,'yyyy-MM') month,
COUNT(*)
FROM db_healthcare_kl.silver.encounter_silver_dlt
GROUP BY month
ORDER BY month;

In [0]:
%sql
SELECT
    COUNT(*)
FROM db_healthcare_kl.silver.encounter_silver_dlt
WHERE date_format(start_date,'yyyy-MM')='2014-03';

In [0]:
%sql
SELECT
    COUNT(*)
FROM db_healthcare_kl.omop.visit_occurrence
WHERE date_format(visit_start_date,'yyyy-MM')='2014-03';

In [0]:
%sql
SELECT
    MAX(total_visits) AS max_visits,
    SUM(total_visits) AS total_visits
FROM db_healthcare_kl.gold.kpi_adt_volume;

In [0]:
%sql
SELECT *
FROM db_healthcare_kl.gold.kpi_adt_volume
ORDER BY total_visits DESC
LIMIT 10;

In [0]:
%sql
SELECT *
FROM db_healthcare_kl.gold.kpi_adt_volume
WHERE total_visits > 100;

In [0]:
end_time = datetime.now()

record_count = bronze_patient.count()

log_pipeline_run(
    pipeline_name="Healthcare Lakehouse",
    layer="Bronze",
    status="SUCCESS",
    records_processed=record_count,
    start_time=start_time,
    end_time=end_time
)